In [1]:
from qiskit_nature.second_q.hamiltonians.lattices import SquareLattice,line_lattice,BoundaryCondition,LineLattice
from qiskit_nature.second_q.hamiltonians import FermiHubbardModel


FermiHubbardModel(lattice=SquareLattice(1,4),onsite_interaction=1.0)
line_lattice = LineLattice(num_nodes=4, boundary_condition=BoundaryCondition.OPEN)
fermi_hubbard_model = FermiHubbardModel(
    line_lattice.uniform_parameters(
        uniform_interaction=1.0,
        uniform_onsite_potential=0.0,
    ),
    onsite_interaction=0.1,
)

fermi_hubbard_model.second_q_op().chop()


/opt/miniconda3/envs/MindSpore/lib/python3.9/site-packages/qiskit_nature/second_q/hamiltonians/lattices/hyper_cubic_lattice.py:298: RuntimeWarning: invalid value encountered in divide
  pi * np.array(divmod(index, size[0])) / (np.array(size)[::-1] - 1)


FermionicOp({'+_0 -_2': 1.0, '-_0 +_2': -1.0, '+_2 -_4': 1.0, '-_2 +_4': -1.0, '+_4 -_6': 1.0, '-_4 +_6': -1.0, '+_1 -_3': 1.0, '-_1 +_3': -1.0, '+_3 -_5': 1.0, '-_3 +_5': -1.0, '+_5 -_7': 1.0, '-_5 +_7': -1.0, '+_0 -_0 +_1 -_1': 0.1, '+_2 -_2 +_3 -_3': 0.1, '+_4 -_4 +_5 -_5': 0.1, '+_6 -_6 +_7 -_7': 0.1}, num_spin_orbitals=8, )

In [3]:
print(fermi_hubbard_model.second_q_op().chop())


Fermionic Operator
number spin orbitals=8, number terms=16
  1.0 * ( +_0 -_2 )
+ -1.0 * ( -_0 +_2 )
+ 1.0 * ( +_2 -_4 )
+ -1.0 * ( -_2 +_4 )
+ 1.0 * ( +_4 -_6 )
+ -1.0 * ( -_4 +_6 )
+ 1.0 * ( +_1 -_3 )
+ -1.0 * ( -_1 +_3 )
+ 1.0 * ( +_3 -_5 )
+ -1.0 * ( -_3 +_5 )
+ 1.0 * ( +_5 -_7 )
+ -1.0 * ( -_5 +_7 )
+ 0.1 * ( +_0 -_0 +_1 -_1 )
+ 0.1 * ( +_2 -_2 +_3 -_3 )
+ 0.1 * ( +_4 -_4 +_5 -_5 )
+ 0.1 * ( +_6 -_6 +_7 -_7 )


In [ ]:
fermi_hubbard_model.second_q_op().

FermionicOp({'+_0 -_2': 1.0, '-_0 +_2': -1.0, '+_2 -_4': 1.0, '-_2 +_4': -1.0, '+_4 -_6': 1.0, '-_4 +_6': -1.0, '+_0 -_0': 0.0, '+_2 -_2': 0.0, '+_4 -_4': 0.0, '+_6 -_6': 0.0, '+_1 -_3': 1.0, '-_1 +_3': -1.0, '+_3 -_5': 1.0, '-_3 +_5': -1.0, '+_5 -_7': 1.0, '-_5 +_7': -1.0, '+_1 -_1': 0.0, '+_3 -_3': 0.0, '+_5 -_5': 0.0, '+_7 -_7': 0.0, '+_0 -_0 +_1 -_1': 0.1, '+_2 -_2 +_3 -_3': 0.1, '+_4 -_4 +_5 -_5': 0.1, '+_6 -_6 +_7 -_7': 0.1}, num_spin_orbitals=8, )

In [ ]:
line_lattice.draw()

In [ ]:
print(fermi_hubbard_model.second_q_op().chop())

In [ ]:
from qiskit_nature.second_q.operators import FermionicOp
from typing import Union
import netket as nk
import netket.experimental as nkx
import numpy as np
import matplotlib.pyplot as plt
import json
def fermionic_op_to_netket(qiskit_op: FermionicOp, hilbert_space: nk.hilbert):
    """
    Convert a Qiskit Nature FermionicOp to a NetKet fermionic operator.
    
    Args:
        qiskit_op: The Qiskit FermionicOp to convert
        hilbert_space: The target NetKet Hilbert space
        
    Returns:
        NetKet fermionic operator equivalent to the input Qiskit operator
    """
    # Initialize empty NetKet operator
    netket_op = 0.0
    
    # Get the terms and coefficients from the Qiskit operator
    terms = qiskit_op.terms()
    
    for term, coeff in terms.items():
        current_term = 1.0
        for label, index in term:
            site = index[0]  # Assuming single index for site
            sz = +1 if label[1] == '+' else -1  # '+' for up, '-' for down
            
            if label[0] == '+':  # Creation operator
                current_term *= nk.operator.fermion.create(hilbert_space, site, sz=sz)
            elif label[0] == '-':  # Annihilation operator
                current_term *= nk.operator.fermion.destroy(hilbert_space, site, sz=sz)
            elif label[0] == 'N':  # Number operator
                current_term *= nk.operator.fermion.number(hilbert_space, site, sz=sz)
            else:
                raise ValueError(f"Unknown operator label: {label}")
        
        netket_op += coeff * current_term
    
    return netket_op

In [ ]:
qiskit_op = fermi_hubbard_model.second_q_op().chop()

# Create NetKet Hilbert space
n_sites = 4  # Should match your lattice size
hilber_space = nk.hilbert.SpinOrbitalFock(n_sites, s=1/2)

# Convert the operator
netket_ham = fermionic_op_to_netket(qiskit_op, hilber_space)
print("Converted Hamiltonian:", netket_ham.operator_string())